# MAGIC Telescope PCA (gamma vs hadron) – Practice Skeleton

**Short name (GitHub):** `PCATel`

**Lab source:** Codecademy *Implementing PCA in Python* on the MAGIC Gamma Telescope (Hillas parameters from CORSIKA air-shower Monte Carlo; Heck et al., FZKA 6019, 1998).

Work this notebook first. Peek at `PCATel_Solution.ipynb` only when stuck. `PCATel.py` holds small helpers. `PCATel_Reusable_Template.ipynb` is the blank pipeline for the next correlated-feature card.

**Files**
- `data/telescope_data.csv` — 19,020 showers × 10 Hillas features + `class` (`g` signal / `h` background)
- `data/data_matrix.csv`, `data/classes.csv` — split copies used in part 2 of the original lab
- `pcatel_flowchart.png` — desired outcome

**Not an IACT trigger, not a discovery paper.** PCA rotates correlated image parameters. A physicist still owns the cut.


## Inline cheat-sheet (keep this cell visible)

See also **`PCATel_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Row / column | row = one air shower; column = one Hillas parameter |
| Standardize | \(x'=(x-\mu)/\sigma\) per column before PCA when units differ |
| Covariance / correlation | \(R = \mathrm{corr}(X)\) (scale-free) or \(S=\mathrm{cov}(X)\) |
| Eigendecomposition | \(R v = \lambda v\); eigenvectors = directions, \(\lambda\) = variance along them |
| Information % | \(100\cdot\lambda_j / \sum_i \lambda_i\) |
| Cumulative | `np.cumsum(percents)` — pick \(k\) so this \(\ge 95\%\) |
| sklearn | `PCA(n_components=k).fit_transform(X_std)` |
| Projection | \(Z = X_\mathrm{std} V_{:,1:k}\) |
| LinearSVC score | `clf.score(X_test, y_test)` after `train_test_split(..., test_size=0.33, random_state=42)` |

**Order:** clean → split label → correlate → eigen / PCA → choose \(k\) → project → (optional) classify → simulate knobs.


## Desired outcome

![flowchart](pcatel_flowchart.png)

1. Load the MAGIC card. Drop NA. Split `class` from the 10 numeric Hillas columns (`data_matrix`).
2. Heatmap the correlation matrix. Expect `fConc` ↔ `fConc1` near 0.98 and a size/length/width block.
3. Eigendecompose the correlation matrix with NumPy. Sort \(\lambda\) descending. Draw the scree and the cumulative-% plot.
4. Standardize columns. Fit `sklearn.decomposition.PCA`. Confirm ratios match the NumPy path.
5. Project onto 2 PCs. Scatter with hue = class.
6. Train LinearSVC on 2 PCs vs the first two *original* standardized columns. Compare hold-out accuracy.
7. Alternates, more practice, simulation knobs (\(k\), \(n\), noise, scale on/off).


## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.decomposition import PCA
    from sklearn.svm import LinearSVC
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    SKLEARN_OK = True
except ImportError:
    SKLEARN_OK = False
    print("scikit-learn is missing — NumPy path still runs; install scikit-learn for Tasks 7–14.")

from PCATel import HILLAS, load_telescope, standardize, eigen_from_corr

np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (7.2, 4.4)


## 1. Why PCA on MAGIC?

Each row is one extensive air shower imaged by the MAGIC telescope. The 10 Hillas parameters describe the ellipse fitted to the Cherenkov image. They are **correlated** (`fLength` with `fWidth` with `fSize`; `fConc` with `fConc1`), so a 10-D classifier spends capacity on redundant axes.

PCA finds an orthogonal rotation that packs that shared variance into the first few principal components. The label (`g` = gamma ray / signal, `h` = hadron / background) is held out of the rotation — PCA is unsupervised.

**Task 1.** In a sentence: why is a correlation heatmap the right first picture, before any eigendecomposition?


In [ ]:
# Task 1 — write the sentence as a comment.


## 2. Observing the dataset (lab Tasks 1–2)

- Remove any nulls.
- Extract numeric columns as `data_matrix` and the label as `classes`.


In [ ]:
# Task 2.1 — read data/telescope_data.csv (index_col=0), dropna, print shape / class counts / head.
# df = ...

# Task 2.2 — classes = df['class']; data_matrix = df.drop(columns='class')


## 3. Correlation heatmap (lab Task 3)

Use `.corr()` on `data_matrix`. `sns.heatmap` or `plt.imshow` are both fine.


In [ ]:
# Task 3 — correlation_matrix = data_matrix.corr()
# Draw a heatmap. Which pair is near |r| = 0.98?


## 4. Eigendecomposition of the correlation matrix (lab Task 4)

`np.linalg.eig` does not sort. Order with `argsort()[::-1]` and apply the same index to both eigenvalues and eigenvector columns.


In [ ]:
# Task 4
# eigenvalues, eigenvectors = np.linalg.eig(correlation_matrix)
# indices = eigenvalues.argsort()[::-1]
# eigenvalues = eigenvalues[indices]
# eigenvectors = eigenvectors[:, indices]
# print lengths vs n_features


## 5. Scree — percent of information per axis (lab Task 5)

`information_proportions = eigenvalues / eigenvalues.sum()`
`information_percents = information_proportions * 100`


In [ ]:
# Task 5 — compute information_percents and plot a scree ('ro-').


## 6. Cumulative information (lab Task 6)

`np.cumsum`. Draw a 95% horizontal line. On *this* card the 95% crossing is near **PC7**, not PC3 — the original Codecademy bean example was a different matrix.


In [ ]:
# Task 6 — cumulative_information_percents = np.cumsum(information_percents)
# plt.hlines(y=95, ...); mark the smallest k with cum >= 95.


## 7. Standardize, then sklearn PCA (lab Tasks 7–10)

PCA on raw Hillas numbers would let `fDist` (~200) dominate `fConc` (~0.4). Subtract the column mean and divide by the column std.

Then:
- `pca = PCA(); principal_components = pca.fit_transform(data_matrix_standardized)`
- eigenvalues from singular values: `pca.singular_values_ ** 2`
- eigenvectors: `pca.components_.T` (sklearn stores them as rows)
- variance ratios: `pca.explained_variance_ratio_`


In [ ]:
# Task 7.1 — mean, sttd, data_matrix_standardized = (data_matrix - mean) / sttd
# Task 7.2 — PCA(); fit_transform; print shapes
# Task 7.3 — singular_values ** 2; components_.T
# Task 7.4 — explained_variance_ratio_ * 100  (should match Task 5)


## 8. Two-component projection + class scatter (lab Tasks 11–12)

`PCA(n_components=2)`. Build a small DataFrame with `PC1`, `PC2`, `class` and scatter (`sns.lmplot` or `plt.scatter`).


In [ ]:
# Task 8.1 — pca2 = PCA(n_components=2); Z2 = pca2.fit_transform(...)
# Task 8.2 — scatter PC1 vs PC2, hue = classes. Do the two classes separate cleanly?


## 9. LinearSVC on 2 PCs vs first 2 original features (lab Tasks 13–14)

Encode `y = classes.astype('category').cat.codes` (`g`→0, `h`→1).

Same split each time: `test_size=0.33`, `random_state=42`.

1. `X = PCA(n_components=2).fit_transform(data_matrix_standardized)` → LinearSVC → `score_1`
2. `X_original = data_matrix_standardized.iloc[:, [0, 1]]` (`fLength`, `fWidth`) → LinearSVC → `score_2`

Which score is higher? Why is that the expected direction?


In [ ]:
# Task 9.1 — 2-PC LinearSVC hold-out score
# Task 9.2 — first-two-original-features LinearSVC hold-out score
# print both. Majority-class baseline is about 0.648 (always guess g).


## 10. Alternate code that reaches the same idea


In [ ]:
# Alternates — implement at least two:
# A. SVD on the centered/scaled matrix: U, S, Vt = np.linalg.svd(X_std, full_matrices=False)
#    PCs = X_std @ Vt.T ; ratios = S**2 / (S**2).sum()
# B. StandardScaler().fit_transform instead of manual (x-mu)/sigma  (ddof=0 vs ddof=1 — ratios almost identical)
# C. Eigendecompose the covariance of the *standardized* matrix instead of .corr() of the raw matrix
# D. sklearn PCA(n_components=0.95) — let the library pick k for a 95% budget


## 11. More practice


In [ ]:
# P1. How many components for 80%, 90%, 95%, 99%?
# P2. Print the loading vector pca.components_[0] against HILLAS. Which features dominate PC1? PC2?
# P3. Reconstruction MSE vs k: Xhat_k = Z_k @ V[:, :k].T on the standardized matrix. Plot MSE(k).
# P4. Repeat the LinearSVC sweep for k = 1..10. Compare to the full 10-feature score and the majority baseline.
# P5. Drop fConc1 (almost a copy of fConc) and re-fit PCA. Does PC1 change much?


## 12. Simulation — turn the knobs

Write `run_once(k, n, noise_sigma, scale=True, seed=0)` that
1. optionally subsamples `n` rows,
2. optionally adds N(0, noise_sigma) to the (scaled) features,
3. projects to `k` PCs,
4. returns LinearSVC hold-out accuracy.

Sweep one knob at a time. Record what happens to accuracy and to the 2-PC scatter.


In [ ]:
# Simulation knobs: k, n, noise_sigma, scale True/False.
# def run_once(...):
#     ...
#     return acc
#
# for k in range(1, 11):
#     print(k, run_once(k=k, n=None, noise_sigma=0.0))


## Audience rewrite (Jočys checklist + McMurrey types)

| Audience | What they need | One sentence on this card |
|----------|----------------|---------------------------|
| Expert (IACT / astroparticle) | loadings, \(\lambda\) spectrum, residual after \(k\) PCs | PC1 is the size–concentration axis; 7 PCs keep ~96% of Hillas variance. |
| Technician (reconstruction / pipeline) | scale step, `n_components`, file paths, seed | Standardize, `PCA(n_components=7)`, write `Z` next to the event id. |
| Executive (observatory / time allocation) | one number, cost of being wrong | Two PCs already beat the two raw length/width axes on a linear screen (0.74 vs 0.72). |
| Nonspecialist (journalist / visitor gallery) | picture, no jargon | We rotate ten correlated ellipse measurements into a few summary axes so signal-like showers sit apart from background. |

Data literacy: experts can read a scree; executives should see the cumulative bar and the two-score comparison, not the eigenvector matrix.


## What this model can and cannot do

**Can**
- Compress ten correlated Hillas parameters into \(k\) uncorrelated axes that keep a chosen variance budget.
- Give a 2-D picture of whether gamma and hadron images already separate linearly.
- Feed a linear classifier with fewer, orthogonal features.

**Cannot**
- Replace a physics-based MAGIC gamma/hadron cut or a boosted-decision-tree working on the full parameter list.
- Use the class label while building the axes (PCA never sees `y`).
- Guarantee that high-variance directions are the ones that separate signal from background.
- Be cited as evidence of a new particle or a flux measurement.

**Top applications of this pattern:** image-parameter compression in IACT analysis; pre-processing before a linear SVM; noise filtering; multicollinearity cleanup before regression; eigenfaces-style compression.

**Anti-applications:** any setting where the scientifically relevant direction has *small* variance; nonlinear manifolds; production triggering with a hard latency budget and no offline scale/PCA fit.


## Next steps

- Compare PCA + LinearSVC to a LinearSVC on all 10 raw (scaled) features — on this card they converge near \(k=10\) at ~0.79.
- Swap LinearSVC for logistic regression or a shallow random forest; does the 2-PC gap vs raw-2 persist?
- Try kernel PCA or UMAP if the PC1–PC2 scatter looks like overlapping blobs rather than two stripes.
- Read `PCATel_Project_Memo.docx` and rewrite the 2-PC score sentence for the executive column.
